# Ligand Preparation Studio — Decoy Theme v1.2
Scientific core preserved from Robust v1.1; PyQt5 interface restyled to match the decoy-validation application.


In [ ]:
# Ligand Preparation GUI for VS Code / Local Python / VS Code Jupyter
# AutoDock Vina: python.exe -m meeko.cli.mk_prepare_ligand
# AutoDock4: MGLTools python.exe/pythonsh.exe + prepare_ligand4.py + OpenBabel
# Run normally: python ligand_prep_gui_vina_ad4_tabs_live_logs.py
# Run in Jupyter cell: %run ligand_prep_gui_vina_ad4_tabs_live_logs.py

import os
import sys
import re
import csv
import math
import json
import platform
import importlib.metadata as importlib_metadata
import shutil
import zipfile
import traceback
import subprocess
import threading
import tempfile
from pathlib import Path
from datetime import datetime

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdMolDescriptors
    from rdkit.Chem.MolStandardize import rdMolStandardize
except Exception as e:
    Chem = None
    RDKit_IMPORT_ERROR = e
else:
    RDKit_IMPORT_ERROR = None

try:
    import tkinter as tk
    from tkinter import ttk, filedialog, messagebox
    from tkinter.scrolledtext import ScrolledText
except Exception as e:
    tk = None
    TK_IMPORT_ERROR = e
else:
    TK_IMPORT_ERROR = None

# ---------------------------
# Your default Windows paths
# ---------------------------
DEFAULT_VINA_MEEKO_PYTHON = r"C:\Users\USER\AppData\Local\Programs\Python\Python313\python.exe"
DEFAULT_OBABEL = r"C:\Program Files\OpenBabel-3.1.1\obabel.exe"
DEFAULT_MGLTOOLS_PYTHON = r"C:\Program Files (x86)\MGLTools-1.5.7\python.exe"
DEFAULT_PREPARE_LIGAND4 = r"C:\Program Files (x86)\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py"

SUPPORTED_STRUCTURE_EXTS = {".sdf", ".mol", ".mol2", ".pdb"}


def safe_filename(name: str, max_len: int = 120) -> str:
    name = str(name).strip() if name is not None else "ligand"
    if not name or name.lower() == "nan":
        name = "ligand"
    name = re.sub(r"[\\/:*?\"<>|]+", "_", name)
    name = re.sub(r"\s+", "_", name)
    name = "".join(ch for ch in name if ch.isalnum() or ch in "._-")
    return name[:max_len] or "ligand"


def which_any(names):
    for n in names:
        p = shutil.which(n)
        if p:
            return p
    return ""


def existing_or_blank(path: str) -> str:
    return path if path and Path(path).exists() else ""


def guess_vina_python() -> str:
    # Prefer the interpreter currently running the application. In the packaged
    # launcher this is the private runtime where Meeko is installed automatically.
    # The original user-defined Python path remains available as a fallback and
    # can still be overridden from the Tool Paths tab.
    current = str(sys.executable or "").strip()
    if current and Path(current).name.lower() == "pythonw.exe":
        console_python = str(Path(current).with_name("python.exe"))
        if Path(console_python).exists():
            current = console_python
    if current and Path(current).exists() and "mgltools" not in current.lower():
        return current
    if Path(DEFAULT_VINA_MEEKO_PYTHON).exists():
        return DEFAULT_VINA_MEEKO_PYTHON
    return current


def guess_obabel() -> str:
    candidates = [
        DEFAULT_OBABEL,
        which_any(["obabel", "obabel.exe"]),
        r"C:\Program Files (x86)\OpenBabel-3.1.1\obabel.exe",
        r"C:\Program Files\OpenBabel-2.4.1\obabel.exe",
        r"C:\Program Files (x86)\OpenBabel-2.4.1\obabel.exe",
        r"C:\Program Files (x86)\MGLTools-1.5.7\OpenBabel-2.3.2\obabel.exe",
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_OBABEL


def guess_mgl_python() -> str:
    candidates = [
        DEFAULT_MGLTOOLS_PYTHON,
        r"C:\Program Files (x86)\MGLTools-1.5.7\pythonsh.exe",
        r"C:\Program Files\MGLTools-1.5.7\python.exe",
        r"C:\Program Files\MGLTools-1.5.7\pythonsh.exe",
        r"C:\MGLTools-1.5.7\python.exe",
        r"C:\MGLTools-1.5.7\pythonsh.exe",
        which_any(["pythonsh", "pythonsh.exe"]),
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_MGLTOOLS_PYTHON


def guess_prepare_ligand4() -> str:
    candidates = [
        DEFAULT_PREPARE_LIGAND4,
        r"C:\Program Files\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\MGLTools-1.5.7\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\Program Files (x86)\PyRx\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
        r"C:\Program Files\Chimera 1.18\bin\Lib\site-packages\AutoDockTools\Utilities24\prepare_ligand4.py",
    ]
    for c in candidates:
        if c and Path(c).exists():
            return c
    return DEFAULT_PREPARE_LIGAND4


def run_cmd(cmd, timeout=240, cwd=None):
    """Run external command safely without shell. Return ok, stdout, stderr, returncode."""
    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout,
            shell=False,
            cwd=str(cwd) if cwd else None,
        )
        return result.returncode == 0, result.stdout, result.stderr, result.returncode
    except Exception as e:
        return False, "", str(e), -999


def load_structure_mols(path, obabel_path=""):
    """
    Load SDF/MOL/MOL2/PDB ligand structure files robustly.

    First RDKit is used with sanitize=False so files that contain minor
    valence/typing issues can still be selected and loaded. Sanitization and
    standardization are performed later by the normal preparation pipeline.

    If RDKit cannot read the structure and OpenBabel is available, the input
    is converted temporarily to SDF and read again. No converted structure
    file is kept in the final output folder.
    """
    path = Path(path)
    ext = path.suffix.lower()
    mols = []

    try:
        if ext == ".sdf":
            suppl = Chem.SDMolSupplier(
                str(path),
                removeHs=False,
                sanitize=False,
                strictParsing=False,
            )
            mols = [m for m in suppl if m is not None]

        elif ext == ".mol":
            m = Chem.MolFromMolFile(
                str(path),
                removeHs=False,
                sanitize=False,
                strictParsing=False,
            )
            mols = [m] if m is not None else []

        elif ext == ".mol2":
            m = Chem.MolFromMol2File(
                str(path),
                removeHs=False,
                sanitize=False,
                cleanupSubstructures=True,
            )
            mols = [m] if m is not None else []

        elif ext == ".pdb":
            m = Chem.MolFromPDBFile(
                str(path),
                removeHs=False,
                sanitize=False,
                proximityBonding=True,
            )
            mols = [m] if m is not None else []
    except Exception:
        mols = []

    if mols:
        return mols

    # Optional OpenBabel fallback for structure files RDKit cannot parse directly.
    obabel = str(obabel_path or "").strip()
    if obabel and Path(obabel).exists() and ext in SUPPORTED_STRUCTURE_EXTS:
        try:
            with tempfile.TemporaryDirectory(prefix="ligand_read_") as td:
                converted = Path(td) / f"{safe_filename(path.stem)}_converted.sdf"
                ok, out, err, code = run_cmd(
                    [obabel, str(path), "-O", str(converted)],
                    timeout=120,
                )
                if ok and converted.exists() and converted.stat().st_size > 0:
                    suppl = Chem.SDMolSupplier(
                        str(converted),
                        removeHs=False,
                        sanitize=False,
                        strictParsing=False,
                    )
                    mols = [m for m in suppl if m is not None]
                    if mols:
                        return mols
        except Exception:
            pass

    return []


def canonical_smiles(mol):
    try:
        m = Chem.RemoveHs(Chem.Mol(mol), sanitize=False)
        Chem.SanitizeMol(m)
        return Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    except Exception:
        try:
            return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        except Exception:
            return ""


def undefined_stereo_count(mol):
    try:
        chiral = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
        return sum(1 for _, label in chiral if label == "?")
    except Exception:
        return 0


def formal_charge(mol):
    try:
        return int(sum(a.GetFormalCharge() for a in mol.GetAtoms()))
    except Exception:
        return 0


def validate_pdbqt(path):
    """Basic file-integrity checks for a generated ligand PDBQT.

    This verifies file presence, atom records, finite Cartesian coordinates,
    TORSDOF, and balanced BRANCH/ENDBRANCH records. It is intentionally an
    integrity check rather than a claim of chemical or docking validity.
    """
    try:
        p = Path(path)
        if not p.exists() or p.stat().st_size < 80:
            return False, "PDBQT missing or too small"
        txt = p.read_text(errors="ignore")
        lines = txt.splitlines()
        atom_lines = [ln for ln in lines if ln.startswith(("ATOM", "HETATM"))]
        if not atom_lines:
            return False, "PDBQT lacks ATOM/HETATM records"

        # PDB/PDBQT coordinates occupy the conventional fixed-width x/y/z fields.
        for ln in atom_lines:
            try:
                x = float(ln[30:38]); y = float(ln[38:46]); z = float(ln[46:54])
            except Exception:
                return False, "PDBQT contains an atom record with unreadable coordinates"
            if not all(math.isfinite(v) for v in (x, y, z)):
                return False, "PDBQT contains non-finite atom coordinates"

        tors = [ln for ln in lines if ln.startswith("TORSDOF")]
        if not tors:
            return False, "PDBQT lacks TORSDOF record"
        try:
            torsdof = int(tors[-1].split()[-1])
            if torsdof < 0:
                return False, "PDBQT has a negative TORSDOF value"
        except Exception:
            return False, "PDBQT has an unreadable TORSDOF value"

        n_branch = sum(1 for ln in lines if ln.startswith("BRANCH"))
        n_endbranch = sum(1 for ln in lines if ln.startswith("ENDBRANCH"))
        if n_branch != n_endbranch:
            return False, f"PDBQT BRANCH/ENDBRANCH mismatch ({n_branch} vs {n_endbranch})"

        return True, "OK"
    except Exception as e:
        return False, str(e)


class LigandPreparator:
    def __init__(self, settings, log_func=None, progress_func=None, counters_func=None):
        self.settings = settings
        self.log = log_func or print
        self.progress = progress_func or (lambda current, total, name="": None)
        self.counters = counters_func or (lambda success, failed, dup: None)

        self.out_dir = Path(settings["output_dir"]).resolve()
        self.out_dir.mkdir(parents=True, exist_ok=True)

        # Final user-visible outputs only:
        #   output_dir/PDBQT/*.pdbqt
        #   output_dir/ligands.txt
        self.pdbqt_dir = self.out_dir / "PDBQT"
        self.pdbqt_dir.mkdir(parents=True, exist_ok=True)

        # SDF/MOL2 preparation files remain temporary and are deleted automatically.
        self._work_root = Path(tempfile.mkdtemp(prefix="ligand_prep_work_"))
        self.sdf_dir = self._work_root / "prepared_3D_SDF"
        self.intermediate_dir = self._work_root / "intermediate_MOL2"
        self.sdf_dir.mkdir(parents=True, exist_ok=True)
        self.intermediate_dir.mkdir(parents=True, exist_ok=True)

        self.rows = []
        self.skipped_duplicates = []
        self.seen_smiles = set()
        self.used_output_names = set()
        self.success_count = 0
        self.failed_count = 0

    def load_inputs(self):
        mode = self.settings["input_mode"]
        records = []

        if mode == "SMILES text":
            raw = self.settings.get("smiles_text", "")
            items = []
            for line in raw.replace(";", "\n").splitlines():
                line = line.strip()
                if not line:
                    continue
                if "," in line:
                    smi, name = line.split(",", 1)
                else:
                    parts = line.split()
                    smi = parts[0]
                    name = "_".join(parts[1:]) if len(parts) > 1 else f"ligand_{len(items)+1}"
                items.append((smi.strip(), name.strip()))
            for smi, name in items:
                records.append({"name": name, "smiles": smi, "mol": None, "source": "SMILES text"})

        elif mode == "CSV with SMILES":
            if pd is None:
                raise RuntimeError("pandas is not installed. Install pandas first.")
            path = Path(self.settings["input_path"])
            df = pd.read_csv(path)
            smiles_col = self.settings.get("smiles_col", "").strip()
            name_col = self.settings.get("name_col", "").strip()
            if not smiles_col:
                for c in df.columns:
                    if c.lower().strip() in ["smiles", "smi", "canonical_smiles", "canonical smiles"]:
                        smiles_col = c
                        break
            if not smiles_col or smiles_col not in df.columns:
                raise RuntimeError(f"SMILES column not found. Available columns: {list(df.columns)}")
            if name_col and name_col not in df.columns:
                self.log(f"Name column '{name_col}' not found. Generated names will be used.")
                name_col = ""
            for idx, row in df.iterrows():
                smi = str(row[smiles_col]).strip()
                if not smi or smi.lower() == "nan":
                    continue
                name = str(row[name_col]).strip() if name_col else f"ligand_{idx+1}"
                records.append({"name": name, "smiles": smi, "mol": None, "source": str(path.name)})

        elif mode == "Structure files":
            paths = [Path(p) for p in self.settings.get("input_files", []) if p]
            obabel = self.settings.get("obabel_path", "").strip()

            for fp in paths:
                if fp.suffix.lower() not in SUPPORTED_STRUCTURE_EXTS:
                    self.log(f"  Ignored unsupported file: {fp.name}")
                    continue
                if not fp.exists():
                    self.log(f"  File not found: {fp}")
                    continue

                mols = load_structure_mols(fp, obabel_path=obabel)
                if not mols:
                    self.log(f"  Could not read structure file: {fp.name}")
                    continue

                # For a normal one-ligand-per-file workflow, the ligand name is
                # exactly the selected file name without its extension.
                # If an SDF contains multiple molecules, numbered suffixes are used.
                for i, mol in enumerate(mols, 1):
                    name = fp.stem if len(mols) == 1 else f"{fp.stem}_{i}"
                    records.append({
                        "name": name,
                        "smiles": "",
                        "mol": mol,
                        "source": fp.name,
                    })

        elif mode == "Folder of structures":
            folder = Path(self.settings["input_path"])
            obabel = self.settings.get("obabel_path", "").strip()
            paths = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in SUPPORTED_STRUCTURE_EXTS
            ]

            for fp in sorted(paths, key=lambda p: p.name.lower()):
                mols = load_structure_mols(fp, obabel_path=obabel)
                if not mols:
                    self.log(f"  Could not read structure file: {fp.name}")
                    continue

                for i, mol in enumerate(mols, 1):
                    name = fp.stem if len(mols) == 1 else f"{fp.stem}_{i}"
                    records.append({
                        "name": name,
                        "smiles": "",
                        "mol": mol,
                        "source": fp.name,
                    })
        else:
            raise RuntimeError("Unknown input mode")
        return records

    def _unique_output_name(self, base_name):
        """Prevent different ligands with identical file-safe names from overwriting outputs."""
        base = safe_filename(base_name)
        candidate = base
        idx = 2
        while candidate.lower() in self.used_output_names:
            candidate = safe_filename(f"{base}_{idx}")
            idx += 1
        self.used_output_names.add(candidate.lower())
        if candidate != base:
            self.log(f"  Warning: output name collision detected; renamed '{base}' to '{candidate}'.")
        return candidate

    def standardize_mol(self, rec):
        name = safe_filename(rec["name"])
        source_smiles = rec.get("smiles", "")
        mol = rec.get("mol")

        if mol is None:
            mol = Chem.MolFromSmiles(source_smiles)
            if mol is None:
                raise ValueError("Invalid SMILES")
        else:
            mol = Chem.Mol(mol)

        try:
            Chem.SanitizeMol(mol)
        except Exception as first_error:
            mol.UpdatePropertyCache(strict=False)
            sanitize_status = Chem.SanitizeMol(mol, catchErrors=True)
            if sanitize_status != Chem.SanitizeFlags.SANITIZE_NONE:
                raise ValueError(
                    f"RDKit sanitization failed after fallback (status={int(sanitize_status)}): {first_error}"
                )

        try:
            mol = rdMolStandardize.Cleanup(mol)
        except Exception as e:
            self.log(f"  Warning: RDKit Cleanup failed; original sanitized structure retained. Reason: {e}")

        if self.settings.get("remove_salts", True):
            try:
                mol = rdMolStandardize.FragmentParent(mol)
            except Exception:
                frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=True)
                if frags:
                    mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())

        if self.settings.get("canonical_tautomer", True):
            try:
                enumerator = rdMolStandardize.TautomerEnumerator()
                mol = enumerator.Canonicalize(mol)
            except Exception as e:
                self.log(f"  Warning: canonical tautomer generation failed; current structure retained. Reason: {e}")

        can_smi = canonical_smiles(mol)
        if self.settings.get("deduplicate", True) and can_smi:
            if can_smi in self.seen_smiles:
                self.skipped_duplicates.append({"name": name, "canonical_smiles": can_smi, "source": rec.get("source", "")})
                raise RuntimeError("Duplicate canonical SMILES skipped")
            self.seen_smiles.add(can_smi)

        name = self._unique_output_name(name)
        mol.SetProp("_Name", name)
        mol.SetProp("OriginalName", str(rec["name"]))
        mol.SetProp("Source", str(rec.get("source", "")))
        mol.SetProp("Canonical_SMILES", can_smi)
        return mol, name, can_smi

    def generate_3d(self, mol):
        ff_choice = self.settings.get("force_field", "AUTO: MMFF94 then UFF")
        n_confs = max(1, int(self.settings.get("num_conformers", 20)))
        mol_h = Chem.AddHs(mol, addCoords=True)

        params = AllChem.ETKDGv3()
        params.randomSeed = int(self.settings.get("random_seed", 42))
        params.pruneRmsThresh = 0.5
        params.useSmallRingTorsions = True
        params.useMacrocycleTorsions = True

        conf_ids = list(AllChem.EmbedMultipleConfs(mol_h, numConfs=n_confs, params=params))
        if not conf_ids:
            cid = AllChem.EmbedMolecule(mol_h, randomSeed=int(self.settings.get("random_seed", 42)), useRandomCoords=True)
            if cid < 0:
                raise RuntimeError("3D conformer generation failed")
            conf_ids = [cid]

        if ff_choice == "ETKDG only; no minimization":
            best_cid = conf_ids[0]
            used_ff = "ETKDG only"
        else:
            best_cid, used_ff = self._minimize_and_pick_best(mol_h, conf_ids, ff_choice)

        best = Chem.Mol(mol_h)
        conf = Chem.Conformer(mol_h.GetConformer(best_cid))
        best.RemoveAllConformers()
        best.AddConformer(conf, assignId=True)
        best.SetProp("ForceFieldUsed", used_ff)
        return best, used_ff, len(conf_ids)

    def _minimize_and_pick_best(self, mol, conf_ids, ff_choice):
        if ff_choice.startswith("AUTO"):
            requested = ["MMFF94", "UFF"]
        elif ff_choice in ["MMFF94", "MMFF94s", "UFF"]:
            requested = [ff_choice]
        else:
            requested = ["MMFF94", "UFF"]

        last_error = None
        for ff in requested:
            energies = []
            nonconverged = 0
            try:
                if ff in ["MMFF94", "MMFF94s"]:
                    variant = "MMFF94s" if ff == "MMFF94s" else "MMFF94"
                    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant=variant)
                    if props is None:
                        raise RuntimeError(f"{ff} parameters unavailable")
                    for cid in conf_ids:
                        try:
                            status = AllChem.MMFFOptimizeMolecule(
                                mol, mmffVariant=variant, confId=cid, maxIters=1000
                            )
                            if status != 0:
                                nonconverged += 1
                                continue
                            force_field = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
                            energy = float(force_field.CalcEnergy())
                            if math.isfinite(energy):
                                energies.append((energy, cid))
                        except Exception as e:
                            last_error = e
                elif ff == "UFF":
                    for cid in conf_ids:
                        try:
                            status = AllChem.UFFOptimizeMolecule(mol, confId=cid, maxIters=1000)
                            if status != 0:
                                nonconverged += 1
                                continue
                            force_field = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
                            energy = float(force_field.CalcEnergy())
                            if math.isfinite(energy):
                                energies.append((energy, cid))
                        except Exception as e:
                            last_error = e

                if nonconverged:
                    self.log(
                        f"  Warning: {ff} did not converge within 1000 iterations for "
                        f"{nonconverged}/{len(conf_ids)} conformer(s); those conformers were not used for selection."
                    )
                if energies:
                    energies.sort(key=lambda x: x[0])
                    return energies[0][1], ff
                last_error = RuntimeError(
                    f"No {ff} conformer reached convergence with a finite energy within 1000 iterations"
                )
            except Exception as e:
                last_error = e
                continue
        raise RuntimeError(f"Force-field minimization failed: {last_error}")

    def write_sdf(self, mol3d, name):
        sdf_path = self.sdf_dir / f"{name}.sdf"
        writer = Chem.SDWriter(str(sdf_path))
        writer.write(mol3d)
        writer.close()
        return sdf_path

    def prepare_vina_pdbqt(self, sdf_path, name):
        vina_python = self.settings.get("vina_python", "").strip() or guess_vina_python()
        if not Path(vina_python).exists():
            raise RuntimeError("Vina Meeko Python.exe path is missing or invalid")
        pdbqt_path = self.pdbqt_dir / f"{name}.pdbqt"
        cmd = [vina_python, "-m", "meeko.cli.mk_prepare_ligand", "-i", str(sdf_path), "-o", str(pdbqt_path)]
        ok, out, err, code = run_cmd(cmd, timeout=int(self.settings.get("timeout_sec", 240)))
        if not ok:
            raise RuntimeError(f"Meeko module failed | code={code} | stderr={err.strip()} | stdout={out.strip()}")
        valid, msg = validate_pdbqt(pdbqt_path)
        if not valid:
            raise RuntimeError(f"Invalid PDBQT from Meeko: {msg}")
        return pdbqt_path, "Meeko module: python.exe -m meeko.cli.mk_prepare_ligand"

    def sdf_to_mol2_with_obabel(self, sdf_path, name):
        obabel = self.settings.get("obabel_path", "").strip() or guess_obabel()
        if not Path(obabel).exists():
            raise RuntimeError("OpenBabel obabel.exe path is required for AutoDock4 SDF to MOL2 conversion")
        mol2_path = self.intermediate_dir / f"{name}.mol2"
        cmd = [obabel, str(sdf_path), "-O", str(mol2_path)]
        ok, out, err, code = run_cmd(cmd, timeout=int(self.settings.get("timeout_sec", 240)))
        if not ok or not mol2_path.exists() or mol2_path.stat().st_size < 80:
            raise RuntimeError(f"OpenBabel SDF to MOL2 failed | code={code} | stderr={err.strip()} | stdout={out.strip()}")
        return mol2_path

    def prepare_ad4_pdbqt(self, sdf_path, name):
        mgl_python = self.settings.get("mgl_python", "").strip()
        prepare_ligand4 = self.settings.get("prepare_ligand4_path", "").strip()
        if not mgl_python or not Path(mgl_python).exists():
            raise RuntimeError("MGLTools python.exe/pythonsh.exe path is missing or invalid")
        if not prepare_ligand4 or not Path(prepare_ligand4).exists():
            raise RuntimeError("prepare_ligand4.py path is missing or invalid")

        # First make a real MOL2 file using OpenBabel.
        mol2_path = self.sdf_to_mol2_with_obabel(sdf_path, name)
        final_pdbqt_path = self.pdbqt_dir / f"{name}.pdbqt"

        # IMPORTANT FIX FOR WINDOWS/MGLTOOLS:
        # AutoDockTools/MolKit sometimes fails when -l/-o are long absolute Windows paths
        # or when output folders contain spaces. Therefore we run prepare_ligand4.py
        # inside a short temporary working folder and pass only local filenames.
        # Keep AutoDock4 working files temporary. Final output remains only PDBQT.
        ad4_temp_root = self._work_root / "AD4_LIGPREP_TEMP"
        lig_work = ad4_temp_root / safe_filename(name)
        lig_work.mkdir(parents=True, exist_ok=True)

        local_mol2 = lig_work / f"{name}.mol2"
        local_pdbqt = lig_work / f"{name}.pdbqt"
        shutil.copy2(mol2_path, local_mol2)

        # -A hydrogens: add hydrogens if missing
        # -U nphs_lps: merge nonpolar hydrogens and lone pairs; standard AD4 ligand cleanup
        cmd = [
            mgl_python,
            prepare_ligand4,
            "-l", local_mol2.name,
            "-o", local_pdbqt.name,
            "-A", "hydrogens",
            "-U", "nphs_lps",
        ]
        ok, out, err, code = run_cmd(
            cmd,
            timeout=int(self.settings.get("timeout_sec", 240)),
            cwd=lig_work,
        )
        if not ok:
            raise RuntimeError(
                f"prepare_ligand4.py failed | code={code} | working_dir={lig_work} | "
                f"stderr={err.strip()} | stdout={out.strip()}"
            )
        if not local_pdbqt.exists() or local_pdbqt.stat().st_size < 80:
            raise RuntimeError(f"prepare_ligand4.py ran but did not create PDBQT in {lig_work}")

        shutil.copy2(local_pdbqt, final_pdbqt_path)
        valid, msg = validate_pdbqt(final_pdbqt_path)
        if not valid:
            raise RuntimeError(f"Invalid PDBQT from prepare_ligand4.py: {msg}")
        return final_pdbqt_path, "AutoDockTools prepare_ligand4.py"

    def process_all(self):
        if Chem is None:
            raise RuntimeError(f"RDKit import failed: {RDKit_IMPORT_ERROR}")
        records = self.load_inputs()
        total = len(records)
        if total == 0:
            raise RuntimeError("No valid input molecule found")

        self.log(f"Loaded {total} input molecule(s).")
        mode = self.settings.get("docking_mode", "AutoDock Vina")

        for i, rec in enumerate(records, 1):
            base_name = safe_filename(rec.get("name", f"ligand_{i}"))
            self.progress(i - 1, total, base_name)
            self.counters(self.success_count, self.failed_count, len(self.skipped_duplicates))
            self.log(f"\n[{i}/{total}] Processing: {base_name}")
            row = {
                "input_index": i,
                "name": base_name,
                "source": rec.get("source", ""),
                "docking_mode": mode,
                "status": "FAILED",
                "reason": "",
                "canonical_smiles": "",
                "formal_charge": "",
                "rotatable_bonds": "",
                "heavy_atoms": "",
                "undefined_stereo_centers": "",
                "force_field_used": "",
                "conformers_generated": "",
                "sdf_path": "",
                "pdbqt_path": "",
                "method": "",
            }
            try:
                mol, name, can_smi = self.standardize_mol(rec)
                row["name"] = name
                row["canonical_smiles"] = can_smi
                row["formal_charge"] = formal_charge(mol)
                row["rotatable_bonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
                row["heavy_atoms"] = mol.GetNumHeavyAtoms()
                row["undefined_stereo_centers"] = undefined_stereo_count(mol)

                if row["undefined_stereo_centers"]:
                    self.log(f"  Warning: {row['undefined_stereo_centers']} undefined stereocenter(s).")
                if row["rotatable_bonds"] != "" and int(row["rotatable_bonds"]) > 15:
                    self.log(f"  Warning: high rotatable bonds = {row['rotatable_bonds']}.")

                mol3d, used_ff, n_confs = self.generate_3d(mol)
                row["force_field_used"] = used_ff
                row["conformers_generated"] = n_confs
                sdf_path = self.write_sdf(mol3d, name)
                row["sdf_path"] = str(sdf_path)
                self.log(f"  Temporary 3D SDF prepared: {sdf_path.name} | FF={used_ff} | conformers={n_confs}")

                if mode == "AutoDock Vina":
                    pdbqt_path, method = self.prepare_vina_pdbqt(sdf_path, name)
                else:
                    pdbqt_path, method = self.prepare_ad4_pdbqt(sdf_path, name)

                row["pdbqt_path"] = str(pdbqt_path)
                row["method"] = method
                row["status"] = "SUCCESS"
                row["reason"] = "OK"
                self.success_count += 1
                self.log(f"  SUCCESS: {pdbqt_path.name}")
            except Exception as e:
                reason = str(e)
                row["reason"] = reason
                if "Duplicate canonical SMILES skipped" in reason:
                    row["status"] = "SKIPPED_DUPLICATE"
                    self.log(f"  SKIPPED DUPLICATE: {base_name}")
                else:
                    row["status"] = "FAILED"
                    self.failed_count += 1
                    self.log(f"  FAILED: {reason}")
            self.rows.append(row)
            self.progress(i, total, base_name)
            self.counters(self.success_count, self.failed_count, len(self.skipped_duplicates))

        try:
            return self.write_reports()
        finally:
            # Remove temporary prepared SDF/MOL2/AutoDock4 working files.
            shutil.rmtree(self._work_root, ignore_errors=True)

    def write_reports(self):
        success_rows = [r for r in self.rows if r["status"] == "SUCCESS"]
        failed_rows = [r for r in self.rows if r["status"] == "FAILED"]
        skipped_rows = [r for r in self.rows if r["status"] == "SKIPPED_DUPLICATE"]

        # User requested only ligand names in ligands.txt and final PDBQT files.
        ligand_list = self.out_dir / "ligands.txt"
        with open(ligand_list, "w", encoding="utf-8") as f:
            for r in success_rows:
                f.write(f"{r['name']}\n")

        self.log("\n==============================")
        self.log("PROCESSING COMPLETE")
        self.log(f"Successful PDBQT: {len(success_rows)}")
        self.log(f"Failed: {len(failed_rows)}")
        self.log(f"Skipped duplicate: {len(skipped_rows)}")
        self.log(f"PDBQT folder: {self.pdbqt_dir}")
        # Structured audit trail for reproducibility and failed-molecule review.
        report_csv = self.out_dir / "processing_report.csv"
        fieldnames = list(self.rows[0].keys()) if self.rows else []
        if fieldnames:
            with open(report_csv, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(self.rows)

        def _pkg_version(name):
            try:
                return importlib_metadata.version(name)
            except Exception:
                return "unknown"

        manifest = {
            "application": "Ligand Preparation Studio",
            "application_version": "1.1",
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "python": sys.version.split()[0],
            "platform": platform.platform(),
            "packages": {
                "rdkit": _pkg_version("rdkit"),
                "meeko": _pkg_version("meeko"),
                "pandas": _pkg_version("pandas"),
                "numpy": _pkg_version("numpy"),
            },
            "settings": {
                "docking_mode": self.settings.get("docking_mode"),
                "input_mode": self.settings.get("input_mode"),
                "force_field": self.settings.get("force_field"),
                "num_conformers": self.settings.get("num_conformers"),
                "random_seed": self.settings.get("random_seed"),
                "remove_salts": self.settings.get("remove_salts"),
                "canonical_tautomer": self.settings.get("canonical_tautomer"),
                "deduplicate": self.settings.get("deduplicate"),
                "timeout_sec": self.settings.get("timeout_sec"),
            },
            "summary": {
                "total": len(self.rows),
                "success": len(success_rows),
                "failed": len(failed_rows),
                "duplicates": len(skipped_rows),
            },
        }
        manifest_json = self.out_dir / "run_manifest.json"
        manifest_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        self.log(f"Ligand names TXT: {ligand_list}")
        self.log(f"Processing report CSV: {report_csv}")
        self.log(f"Run manifest JSON: {manifest_json}")
        if failed_rows:
            self.log("Failed ligand details are preserved in processing_report.csv and the live log.")
        self.log("==============================")

        return {
            "total": len(self.rows),
            "success": len(success_rows),
            "failed": len(failed_rows),
            "duplicates": len(skipped_rows),
            "pdbqt_dir": str(self.pdbqt_dir),
            "ligands_txt": str(ligand_list),
            "processing_report": str(report_csv),
            "run_manifest": str(manifest_json),
        }




USER_MANUAL_HTML = r"""
<h2>Ligand Preparation Studio — User Manual</h2>
<p><b>Purpose</b><br>
This application prepares ligand structures for AutoDock Vina or AutoDock4 while keeping the original scientific preparation workflow unchanged.</p>

<h3>1. Select the docking-preparation mode</h3>
<ul>
<li><b>AutoDock Vina / Vina 1.2:</b> prepares ligand PDBQT files using Meeko.</li>
<li><b>AutoDock4:</b> prepares ligand PDBQT files using OpenBabel, MGLTools and <code>prepare_ligand4.py</code>.</li>
</ul>

<h3>2. Select ligand input</h3>
<p>The application supports:</p>
<ul>
<li><b>CSV with SMILES</b> — select the CSV file and provide the SMILES column and compound-name/ID column.</li>
<li><b>SMILES text</b> — enter SMILES directly as <code>SMILES</code> or <code>SMILES,Name</code>.</li>
<li><b>Structure files</b> — select individual SDF, MOL, MOL2 or PDB files.</li>
<li><b>Folder of structures</b> — select a folder containing supported structure files.</li>
</ul>

<h3>3. Standardization and 3D generation</h3>
<ul>
<li>Select the force field and requested number of conformers.</li>
<li>Set the random seed for reproducible conformer generation.</li>
<li>Optional settings allow salt/counterion removal, canonical tautomer generation and duplicate-SMILES removal.</li>
</ul>

<h3>4. Configure external tool paths</h3>
<p>Open the <b>Tool Paths</b> tab and provide the paths required for the selected docking mode.</p>
<ul>
<li><b>AutoDock Vina:</b> Meeko-compatible Python executable.</li>
<li><b>AutoDock4:</b> OpenBabel <code>obabel.exe</code>, MGLTools Python/Pythonsh and <code>prepare_ligand4.py</code>.</li>
</ul>
<p>Use <b>Test Selected Tool Paths</b> before processing when setting up a new computer.</p>

<h3>5. Select the output directory</h3>
<p>Open the <b>Output</b> tab and choose the folder where the final prepared files should be saved.</p>

<h3>6. Run ligand preparation</h3>
<p>Click <b>RUN LIGAND PREPARATION</b>. The application automatically switches to the live-log page and displays:</p>
<ul>
<li>current ligand and total progress,</li>
<li>successful preparations,</li>
<li>failed preparations,</li>
<li>skipped duplicate structures, and</li>
<li>detailed processing messages.</li>
</ul>

<h3>7. Main outputs</h3>
<ul>
<li><b>PDBQT/</b> — final prepared ligand PDBQT files.</li>
<li><b>ligands.txt</b> — names of successfully prepared ligands.</li>
<li><b>processing_report.csv</b> — per-ligand preparation/QC status and failure reasons.</li>
<li><b>run_manifest.json</b> — run settings and software-version information.</li>
</ul>
<p>Temporary SDF and MOL2 working files are removed automatically after processing.</p>

<h3>Important notes</h3>
<ul>
<li>Verify the selected docking mode and external-tool paths before starting a large batch.</li>
<li>Inspect the processing report for failed or skipped compounds.</li>
<li>Prepared ligand files are computational inputs and should be checked before downstream docking.</li>
</ul>
"""


# =============================================================================
# PyQt5 professional interface — decoy-validator visual theme
# Scientific preparation core above is unchanged.
# =============================================================================

def launch_qt_gui():
    try:
        from PyQt5.QtCore import Qt, QThread, pyqtSignal
        from PyQt5.QtWidgets import (
            QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
            QGridLayout, QLabel, QPushButton, QFileDialog, QMessageBox,
            QTabWidget, QTextEdit, QGroupBox, QLineEdit, QComboBox,
            QSpinBox, QCheckBox, QProgressBar, QFrame, QRadioButton,
            QButtonGroup, QDialog, QDialogButtonBox, QAction
        )
    except Exception as e:
        raise RuntimeError('PyQt5 is required for the professional GUI. Install with: python -m pip install PyQt5') from e

    class PrepWorker(QThread):
        log_signal = pyqtSignal(str)
        progress_signal = pyqtSignal(int, int, str)
        counters_signal = pyqtSignal(int, int, int)
        completed = pyqtSignal(object)
        failed = pyqtSignal(str)

        def __init__(self, settings):
            super().__init__()
            self.settings = settings

        def run(self):
            try:
                prep = LigandPreparator(
                    self.settings,
                    log_func=lambda msg: self.log_signal.emit(str(msg)),
                    progress_func=lambda cur, total, name='': self.progress_signal.emit(int(cur), int(total), str(name)),
                    counters_func=lambda suc, fail, dup: self.counters_signal.emit(int(suc), int(fail), int(dup)),
                )
                result = prep.process_all()
                self.completed.emit(result)
            except Exception:
                self.failed.emit(traceback.format_exc())

    class TextDialog(QDialog):
        def __init__(self, title, html, parent=None, width=900, height=700):
            super().__init__(parent)
            self.setWindowTitle(title)
            self.resize(width, height)
            layout = QVBoxLayout(self)
            browser = QTextEdit()
            browser.setReadOnly(True)
            browser.setHtml(html)
            browser.setStyleSheet("font-size: 11pt; padding: 10px;")
            layout.addWidget(browser)
            buttons = QDialogButtonBox(QDialogButtonBox.Close)
            buttons.button(QDialogButtonBox.Close).clicked.connect(self.close)
            layout.addWidget(buttons)


    class App(QMainWindow):
        def __init__(self):
            super().__init__()
            self.setWindowTitle('Ligand Preparation Studio - AutoDock Vina / AutoDock4')
            self.resize(1500, 940)
            self.setMinimumSize(1180, 760)
            self.worker = None
            self.input_files = []
            self.log_lines = []

            self._apply_decoy_theme()
            self._build_menu()
            self._build_ui()
            self._update_input_mode()
            self._update_docking_mode()
            self._append_log('Ready. Select docking mode, ligand input, tool paths, and click Run Ligand Preparation.')
            self._append_log('AutoDock Vina requires a Python environment with Meeko. AutoDock4 requires OpenBabel + MGLTools + prepare_ligand4.py.')

        def _apply_decoy_theme(self):
            self.setStyleSheet('''
            QMainWindow{background:#eef3f8;}
            QWidget{font-family:'Segoe UI';font-size:10.5pt;color:#182230;}
            QGroupBox{font-weight:700;border:1px solid #d7e0ea;border-radius:10px;margin-top:10px;padding:13px 10px 10px 10px;background:white;}
            QGroupBox::title{subcontrol-origin:margin;left:14px;padding:0 6px;color:#17324d;}
            QPushButton{background:#245b9e;color:white;border:none;border-radius:7px;padding:9px 14px;font-weight:700;}
            QPushButton:hover{background:#1b4a83;}
            QPushButton#run{background:#14804a;} QPushButton#run:hover{background:#0f683c;}
            QPushButton#output{background:#6d43b5;} QPushButton#output:hover{background:#583595;}
            QPushButton#secondary{background:#475569;} QPushButton#secondary:hover{background:#334155;}
            QPushButton:disabled{background:#aeb8c5;color:#eef2f6;}
            QLineEdit,QSpinBox,QComboBox{background:white;border:1px solid #cbd6e2;border-radius:6px;padding:7px;min-height:20px;}
            QLineEdit:focus,QSpinBox:focus,QComboBox:focus{border:1px solid #2e73b8;}
            QTabWidget::pane{border:1px solid #d7e0ea;border-radius:8px;background:white;}
            QTabBar::tab{padding:10px 16px;background:#e7edf4;margin-right:2px;border-top-left-radius:5px;border-top-right-radius:5px;}
            QTabBar::tab:selected{background:white;font-weight:700;color:#17324d;}
            QTextEdit{background:white;border:1px solid #d7e0ea;border-radius:7px;}
            QProgressBar{border:1px solid #cbd6e2;border-radius:6px;text-align:center;background:white;min-height:20px;font-weight:700;}
            QProgressBar::chunk{background:#2e73b8;border-radius:5px;}
            QCheckBox,QRadioButton{spacing:7px;}
            ''')

        def _build_menu(self):
            help_menu = self.menuBar().addMenu("Help")
            manual_action = QAction("User Manual", self)
            manual_action.triggered.connect(self.show_user_manual)
            help_menu.addAction(manual_action)
            about_action = QAction("About", self)
            about_action.triggered.connect(self.show_about)
            help_menu.addAction(about_action)

        def _build_ui(self):
            root = QWidget(); self.setCentralWidget(root)
            main = QVBoxLayout(root); main.setContentsMargins(16,16,16,14); main.setSpacing(11)

            header = QFrame()
            header.setStyleSheet('QFrame{background:qlineargradient(x1:0,y1:0,x2:1,y2:0,stop:0 #0f2742,stop:0.58 #173b61,stop:1 #245b9e);border-radius:12px;}')
            hl = QVBoxLayout(header); hl.setContentsMargins(22,17,22,17); hl.setSpacing(5)
            title = QLabel('Ligand Preparation Studio'); title.setAlignment(Qt.AlignCenter)
            title.setStyleSheet('color:white;font-size:25px;font-weight:800;letter-spacing:0.2px;')
            sub = QLabel('Structure standardization • 3D conformer preparation • AutoDock Vina / AutoDock4 PDBQT generation')
            sub.setAlignment(Qt.AlignCenter); sub.setStyleSheet('color:#dbe8f6;font-size:11.5pt;font-weight:600;')
            dev = QLabel('Developed by Md Fazlay Rabbi  |  Department of Zoology, University of Rajshahi')
            dev.setAlignment(Qt.AlignCenter); dev.setStyleSheet('color:#bcd0e6;font-size:10.5pt;font-weight:600;')
            hl.addWidget(title); hl.addWidget(sub); hl.addWidget(dev); main.addWidget(header)

            self.tabs = QTabWidget(); main.addWidget(self.tabs, 1)
            self.setup_tab = QWidget(); self.tools_tab = QWidget(); self.logs_tab = QWidget(); self.output_tab = QWidget()
            self.tabs.addTab(self.setup_tab, '1  Setup')
            self.tabs.addTab(self.tools_tab, '2  Tool Paths')
            self.tabs.addTab(self.logs_tab, '3  Live Logs / Progress')
            self.tabs.addTab(self.output_tab, '4  Output')
            self._build_setup_tab(); self._build_tools_tab(); self._build_logs_tab(); self._build_output_tab()

            action = QFrame(); action.setStyleSheet('QFrame{background:#ffffff;border:1px solid #d7e0ea;border-radius:9px;}')
            al = QHBoxLayout(action); al.setContentsMargins(12,9,12,9)
            self.run_btn = QPushButton('RUN LIGAND PREPARATION'); self.run_btn.setObjectName('run'); self.run_btn.clicked.connect(self._start_run)
            self.logs_btn = QPushButton('Go to Live Logs'); self.logs_btn.setObjectName('secondary'); self.logs_btn.clicked.connect(lambda: self.tabs.setCurrentWidget(self.logs_tab))
            self.open_output_btn = QPushButton('Open Output Folder'); self.open_output_btn.setObjectName('output'); self.open_output_btn.clicked.connect(self._open_output)
            self.status_label = QLabel('Ready'); self.status_label.setStyleSheet('font-weight:700;color:#476176;')
            al.addWidget(self.run_btn); al.addWidget(self.logs_btn); al.addWidget(self.open_output_btn); al.addSpacing(12); al.addWidget(self.status_label); al.addStretch()
            manual_btn = QPushButton('User Manual'); manual_btn.setObjectName('secondary'); manual_btn.clicked.connect(self.show_user_manual); al.addWidget(manual_btn)
            about_btn = QPushButton('About'); about_btn.setObjectName('secondary'); about_btn.clicked.connect(self.show_about); al.addWidget(about_btn)
            brand = QLabel('Ligand Preparation Studio'); brand.setStyleSheet('color:#708399;font-size:9.5pt;font-weight:600;'); al.addWidget(brand)
            main.addWidget(action)

        def _build_setup_tab(self):
            lay = QVBoxLayout(self.setup_tab); lay.setContentsMargins(14,14,14,14); lay.setSpacing(10)
            mode = QGroupBox('Docking preparation mode'); ml = QHBoxLayout(mode)
            self.vina_radio = QRadioButton('AutoDock Vina / Vina 1.2 ligand PDBQT using Meeko')
            self.ad4_radio = QRadioButton('AutoDock4 ligand PDBQT using AutoDockTools prepare_ligand4.py')
            self.vina_radio.setChecked(True); self.mode_group = QButtonGroup(self); self.mode_group.addButton(self.vina_radio); self.mode_group.addButton(self.ad4_radio)
            self.vina_radio.toggled.connect(self._update_docking_mode); ml.addWidget(self.vina_radio); ml.addSpacing(25); ml.addWidget(self.ad4_radio); ml.addStretch(); lay.addWidget(mode)

            inp = QGroupBox('Input ligands'); gl = QGridLayout(inp)
            self.input_mode = QComboBox(); self.input_mode.addItems(['CSV with SMILES','SMILES text','Structure files','Folder of structures']); self.input_mode.currentTextChanged.connect(self._update_input_mode)
            self.input_path = QLineEdit(); self.input_path.setPlaceholderText('Choose CSV, structure folder, or selected structure files')
            self.input_browse = QPushButton('Browse'); self.input_browse.clicked.connect(self._browse_input)
            self.smiles_col = QLineEdit('smiles'); self.name_col = QLineEdit('Compound_name')
            self.smiles_text = QLineEdit(); self.smiles_text.setPlaceholderText('SMILES or SMILES,Name')
            gl.addWidget(QLabel('Input type'),0,0); gl.addWidget(self.input_mode,0,1,1,2)
            gl.addWidget(QLabel('Input path / files'),1,0); gl.addWidget(self.input_path,1,1); gl.addWidget(self.input_browse,1,2)
            gl.addWidget(QLabel('SMILES column'),2,0); gl.addWidget(self.smiles_col,2,1)
            gl.addWidget(QLabel('Name / ID column'),2,2); gl.addWidget(self.name_col,2,3)
            gl.addWidget(QLabel('SMILES text'),3,0); gl.addWidget(self.smiles_text,3,1,1,3)
            gl.setColumnStretch(1,1); gl.setColumnStretch(3,1); lay.addWidget(inp)

            prep = QGroupBox('Standardization and 3D generation'); pg = QGridLayout(prep)
            self.force_field = QComboBox(); self.force_field.addItems(['MMFF94','MMFF94s','UFF','AUTO: MMFF94 then UFF','ETKDG only; no minimization'])
            self.num_confs = QSpinBox(); self.num_confs.setRange(1,200); self.num_confs.setValue(20)
            self.seed = QSpinBox(); self.seed.setRange(1,999999); self.seed.setValue(42)
            self.remove_salts = QCheckBox('Remove salts / counterions and retain parent fragment'); self.remove_salts.setChecked(True)
            self.canonical_tautomer = QCheckBox('Generate canonical tautomer using RDKit'); self.canonical_tautomer.setChecked(True)
            self.deduplicate = QCheckBox('Skip duplicate canonical isomeric SMILES'); self.deduplicate.setChecked(True)
            pg.addWidget(QLabel('Force field'),0,0); pg.addWidget(self.force_field,0,1)
            pg.addWidget(QLabel('Requested conformers'),0,2); pg.addWidget(self.num_confs,0,3)
            pg.addWidget(QLabel('Random seed'),0,4); pg.addWidget(self.seed,0,5)
            pg.addWidget(self.remove_salts,1,0,1,3); pg.addWidget(self.canonical_tautomer,1,3,1,3); pg.addWidget(self.deduplicate,2,0,1,6)
            note=QLabel('Scientific preparation logic is unchanged; only the interface is restyled to match the decoy application.')
            note.setStyleSheet('color:#59677a;font-style:italic;'); pg.addWidget(note,3,0,1,6); lay.addWidget(prep); lay.addStretch()

        def _build_tools_tab(self):
            lay=QVBoxLayout(self.tools_tab); lay.setContentsMargins(14,14,14,14); lay.setSpacing(10)
            box=QGroupBox('External tool locations'); g=QGridLayout(box)
            self.vina_python=QLineEdit(guess_vina_python())
            self.obabel_path=QLineEdit(guess_obabel())
            self.mgl_python=QLineEdit(guess_mgl_python())
            self.prepare4_path=QLineEdit(guess_prepare_ligand4())
            self.timeout=QSpinBox(); self.timeout.setRange(30,3600); self.timeout.setValue(240)
            rows=[
                ('Vina Meeko Python.exe',self.vina_python,'Python used as: python.exe -m meeko.cli.mk_prepare_ligand','python'),
                ('OpenBabel obabel.exe',self.obabel_path,'Required for AutoDock4 SDF → MOL2 conversion','exe'),
                ('MGLTools python.exe / pythonsh.exe',self.mgl_python,'AutoDock4 preparation environment','python'),
                ('prepare_ligand4.py',self.prepare4_path,'AutoDockTools ligand preparation script','py'),
            ]
            self.tool_rows=[]
            for r,(lab,edit,note,kind) in enumerate(rows):
                l=QLabel(lab); b=QPushButton('Browse'); b.clicked.connect(lambda _, e=edit, k=kind: self._browse_tool(e,k)); n=QLabel(note); n.setStyleSheet('color:#59677a;')
                g.addWidget(l,r,0); g.addWidget(edit,r,1); g.addWidget(b,r,2); g.addWidget(n,r,3); self.tool_rows.append((l,edit,b,n))
            g.addWidget(QLabel('Timeout / ligand (sec)'),4,0); g.addWidget(self.timeout,4,1)
            test=QPushButton('Test Selected Tool Paths'); test.setObjectName('secondary'); test.clicked.connect(self._test_tools); g.addWidget(test,4,2)
            g.setColumnStretch(1,1); lay.addWidget(box)
            info=QGroupBox('Mode requirements'); il=QVBoxLayout(info)
            t=QLabel('AutoDock Vina: uses the selected Meeko Python.exe.\nAutoDock4: uses OpenBabel obabel.exe + MGLTools python.exe/pythonsh.exe + prepare_ligand4.py.')
            t.setStyleSheet('color:#17324d;font-weight:600;'); il.addWidget(t); lay.addWidget(info); lay.addStretch()

        def _build_logs_tab(self):
            lay=QVBoxLayout(self.logs_tab); lay.setContentsMargins(14,14,14,14); lay.setSpacing(10)
            stat=QGroupBox('Running status'); g=QGridLayout(stat)
            self.current_val=QLabel('0'); self.total_val=QLabel('0'); self.success_val=QLabel('0'); self.failed_val=QLabel('0'); self.dup_val=QLabel('0'); self.current_name=QLabel('Ready')
            for w in [self.current_val,self.total_val,self.success_val,self.failed_val,self.dup_val]: w.setStyleSheet('font-size:13pt;font-weight:800;color:#17324d;')
            labels=[('Current',self.current_val),('Total',self.total_val),('Success',self.success_val),('Failed',self.failed_val),('Duplicates',self.dup_val)]
            for i,(lab,val) in enumerate(labels): g.addWidget(QLabel(lab),0,i); g.addWidget(val,1,i)
            g.addWidget(QLabel('Current molecule'),2,0); g.addWidget(self.current_name,2,1,1,4)
            self.progress=QProgressBar(); self.progress.setRange(0,100); self.progress.setValue(0); g.addWidget(self.progress,3,0,1,5); lay.addWidget(stat)
            self.log=QTextEdit(); self.log.setReadOnly(True); lay.addWidget(self.log,1)
            row=QHBoxLayout(); save=QPushButton('Save Log as TXT'); save.clicked.connect(self._save_log); out=QPushButton('Open Output Folder'); out.setObjectName('output'); out.clicked.connect(self._open_output); row.addWidget(save); row.addWidget(out); row.addStretch(); lay.addLayout(row)

        def _build_output_tab(self):
            lay=QVBoxLayout(self.output_tab); lay.setContentsMargins(14,14,14,14); lay.setSpacing(10)
            box=QGroupBox('Output files'); g=QGridLayout(box)
            self.output_dir=QLineEdit(str(Path.cwd()/'ligand_prep_output')); choose=QPushButton('Choose Location'); choose.setObjectName('secondary'); choose.clicked.connect(self._browse_output)
            openb=QPushButton('Open Output Folder'); openb.setObjectName('output'); openb.clicked.connect(self._open_output)
            g.addWidget(QLabel('Export all files to your selected location'),0,0,1,3)
            g.addWidget(self.output_dir,1,0,1,2); g.addWidget(choose,1,2); g.addWidget(openb,1,3); lay.addWidget(box)
            note=QGroupBox('Generated outputs'); nl=QVBoxLayout(note)
            txt=QLabel('PDBQT/  → final prepared ligand files\nligands.txt  → successfully prepared ligand names\nprocessing_report.csv  → per-ligand QC/status/failure reasons\nrun_manifest.json  → run settings and software-version record\n\nTemporary SDF/MOL2 working files are deleted after processing.')
            txt.setStyleSheet('color:#17324d;'); nl.addWidget(txt); lay.addWidget(note); lay.addStretch()

        def _update_input_mode(self):
            mode=self.input_mode.currentText() if hasattr(self,'input_mode') else 'CSV with SMILES'
            is_csv=mode=='CSV with SMILES'; is_text=mode=='SMILES text'
            self.input_path.setEnabled(not is_text); self.input_browse.setEnabled(not is_text)
            self.smiles_col.setEnabled(is_csv); self.name_col.setEnabled(is_csv); self.smiles_text.setEnabled(is_text)
            if is_text: self.input_path.clear()

        def _update_docking_mode(self):
            if not hasattr(self,'tool_rows'): return
            vina=self.vina_radio.isChecked()
            for i,(lab,edit,btn,note) in enumerate(self.tool_rows):
                enabled = (i==0 and vina) or (i>0 and not vina)
                for w in (lab,edit,btn,note): w.setEnabled(enabled)

        def _browse_input(self):
            mode=self.input_mode.currentText()
            if mode=='CSV with SMILES':
                p,_=QFileDialog.getOpenFileName(self,'Select CSV with SMILES','','CSV files (*.csv);;All files (*)')
                if p: self.input_path.setText(p); self.input_files=[]
            elif mode=='Structure files':
                ps,_=QFileDialog.getOpenFileNames(self,'Select ligand structure files','','Structures (*.sdf *.mol *.mol2 *.pdb);;All files (*)')
                if ps: self.input_files=ps; self.input_path.setText(f'{len(ps)} structure file(s) selected')
            elif mode=='Folder of structures':
                d=QFileDialog.getExistingDirectory(self,'Select structure folder')
                if d: self.input_path.setText(d); self.input_files=[]

        def _browse_output(self):
            d=QFileDialog.getExistingDirectory(self,'Choose output folder')
            if d: self.output_dir.setText(d)

        def _browse_tool(self, edit, kind):
            if kind=='py': flt='Python scripts (*.py);;All files (*)'
            else: flt='Executables (*.exe);;All files (*)'
            p,_=QFileDialog.getOpenFileName(self,'Select tool','',flt)
            if p: edit.setText(p)

        def _test_tools(self):
            try:
                if self.vina_radio.isChecked():
                    p=self.vina_python.text().strip()
                    if not p or not Path(p).exists(): raise RuntimeError('Vina Meeko Python.exe path is missing or invalid.')
                    ok,out,err,code=run_cmd([p,'-c','import meeko; print("Meeko OK")'],timeout=60)
                    if not ok: raise RuntimeError(f'Meeko test failed: {err or out}')
                    QMessageBox.information(self,'Tool test','Vina Meeko Python path is valid and Meeko imported successfully.')
                else:
                    checks=[('OpenBabel',self.obabel_path.text()),('MGLTools Python',self.mgl_python.text()),('prepare_ligand4.py',self.prepare4_path.text())]
                    missing=[name for name,p in checks if not p or not Path(p).exists()]
                    if missing: raise RuntimeError('Missing/invalid: '+', '.join(missing))
                    ok,out,err,code=run_cmd([self.obabel_path.text().strip(),'-V'],timeout=30)
                    if not ok: raise RuntimeError(f'OpenBabel test failed: {err or out}')
                    QMessageBox.information(self,'Tool test','AutoDock4 tool paths exist and OpenBabel responds correctly.')
            except Exception as e:
                QMessageBox.critical(self,'Tool test failed',str(e))

        def _collect_settings(self):
            mode=self.input_mode.currentText(); ipath='' if mode=='SMILES text' else self.input_path.text().strip()
            if mode=='Structure files' and not self.input_files: raise RuntimeError('Select at least one structure file.')
            if mode in ('CSV with SMILES','Folder of structures') and not ipath: raise RuntimeError('Select an input path.')
            if mode=='SMILES text' and not self.smiles_text.text().strip(): raise RuntimeError('Enter at least one SMILES string.')
            out=self.output_dir.text().strip()
            if not out: raise RuntimeError('Select an output folder.')
            return {
                'docking_mode':'AutoDock Vina' if self.vina_radio.isChecked() else 'AutoDock4', 'input_mode':mode,
                'input_path':ipath, 'input_files':list(self.input_files), 'smiles_col':self.smiles_col.text().strip(),
                'name_col':self.name_col.text().strip(), 'smiles_text':self.smiles_text.text(), 'output_dir':out,
                'force_field':self.force_field.currentText(), 'num_conformers':self.num_confs.value(), 'random_seed':self.seed.value(),
                'remove_salts':self.remove_salts.isChecked(), 'canonical_tautomer':self.canonical_tautomer.isChecked(),
                'deduplicate':self.deduplicate.isChecked(), 'vina_python':self.vina_python.text().strip(),
                'obabel_path':self.obabel_path.text().strip(), 'mgl_python':self.mgl_python.text().strip(),
                'prepare_ligand4_path':self.prepare4_path.text().strip(), 'timeout_sec':self.timeout.value(),
            }

        def _start_run(self):
            if self.worker and self.worker.isRunning(): return
            try: settings=self._collect_settings()
            except Exception as e: QMessageBox.warning(self,'Missing or invalid settings',str(e)); return
            self.log.clear(); self.log_lines=[]; self.progress.setValue(0)
            self.current_val.setText('0'); self.total_val.setText('0'); self.success_val.setText('0'); self.failed_val.setText('0'); self.dup_val.setText('0'); self.current_name.setText('Starting...')
            self.run_btn.setEnabled(False); self.status_label.setText('Ligand preparation running...'); self.tabs.setCurrentWidget(self.logs_tab)
            self.worker=PrepWorker(settings)
            self.worker.log_signal.connect(self._append_log); self.worker.progress_signal.connect(self._progress); self.worker.counters_signal.connect(self._counters); self.worker.completed.connect(self._done); self.worker.failed.connect(self._failed); self.worker.start()

        def _append_log(self,msg): self.log_lines.append(str(msg)); self.log.append(str(msg))
        def _progress(self,current,total,name):
            self.current_val.setText(str(current)); self.total_val.setText(str(total)); self.current_name.setText(name or 'Processing')
            self.progress.setValue(int(round(100*current/total)) if total else 0)
        def _counters(self,s,f,d): self.success_val.setText(str(s)); self.failed_val.setText(str(f)); self.dup_val.setText(str(d))
        def _done(self,res):
            self.run_btn.setEnabled(True); self.progress.setValue(100); self.status_label.setText('Preparation completed')
            self._append_log(''); self._append_log(f"Completed: {res.get('success',0)} success, {res.get('failed',0)} failed, {res.get('duplicates',0)} duplicate(s).")
            QMessageBox.information(self,'Processing complete',f"Successful PDBQT: {res.get('success',0)}\nFailed: {res.get('failed',0)}\nDuplicates: {res.get('duplicates',0)}\n\nOutput: {res.get('pdbqt_dir','')}")
        def _failed(self,msg):
            self.run_btn.setEnabled(True); self.status_label.setText('Preparation failed'); self._append_log(msg); QMessageBox.critical(self,'Preparation failed',msg[-3500:])
        def _save_log(self):
            p,_=QFileDialog.getSaveFileName(self,'Save live log','ligand_preparation_log.txt','Text files (*.txt);;All files (*)')
            if p: Path(p).write_text('\n'.join(self.log_lines),encoding='utf-8')
        def show_user_manual(self):
            TextDialog('User Manual', USER_MANUAL_HTML, self, 930, 720).exec_()

        def show_about(self):
            html = """
            <h2>Ligand Preparation Studio</h2>
            <p><b>Version:</b> 1.2</p>
            <p>Ligand standardization, 3D conformer preparation and PDBQT generation for AutoDock Vina and AutoDock4 workflows.</p>
            <p><b>Developed by:</b><br>Md Fazlay Rabbi</p>
            <p><b>Affiliation:</b><br>Department of Zoology, University of Rajshahi, Bangladesh</p>
            <p><b>Contact:</b> rabbi.ms244@gmail.com</p>
            <hr>
            <p><b>Key outputs:</b> prepared PDBQT ligands, ligand list, per-ligand processing report and reproducibility manifest.</p>
            """
            TextDialog('About', html, self, 650, 480).exec_()

        def _open_output(self):
            p=Path(self.output_dir.text().strip() or '.'); p.mkdir(parents=True,exist_ok=True)
            try:
                if os.name=='nt': os.startfile(str(p))
                elif sys.platform=='darwin': subprocess.Popen(['open',str(p)])
                else: subprocess.Popen(['xdg-open',str(p)])
            except Exception as e: QMessageBox.warning(self,'Open folder',str(e))

    app=QApplication.instance() or QApplication(sys.argv)
    app.setApplicationName('Ligand Preparation Studio')
    win=App(); win.show()
    return app.exec_()


if __name__ == '__main__':
    launch_qt_gui()
